<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/Copy_of_Corpus_Pro_Professional_Suite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
import sys
import os
import csv
import json
import re
from collections import Counter

# --- ІМПОРТ БІБЛІОТЕК ---
try:
    from PyQt6.QtWidgets import (
        QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
        QComboBox, QLineEdit, QLabel, QFrame, QPushButton, QTabWidget,
        QTableWidget, QTableWidgetItem, QHeaderView, QFileDialog, QMessageBox,
        QSplitter, QToolTip
    )
    from PyQt6.QtCore import Qt
    from PyQt6.QtGui import QFont
except ImportError:
    print("Помилка: PyQt6 не знайдено.")
    sys.exit(1)

try:
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
    from wordcloud import WordCloud
except ImportError:
    print("Помилка: matplotlib або wordcloud не знайдено.")

# --- РОЗШИРЕНИЙ СПИСОК СТОП-СЛІВ (Stop Words Filtering) ---
STOP_WORDS = {
    "the", "a", "an", "and", "or", "but", "if", "then", "else", "when", "at", "by",
    "from", "for", "with", "in", "on", "to", "of", "is", "it", "this", "that", "was",
    "were", "be", "been", "being", "have", "has", "had", "do", "does", "did", "as",
    "i", "you", "he", "she", "we", "they", "my", "your", "his", "her", "their",
    "which", "who", "whom", "whose", "this", "these", "those", "am", "are", "been"
}

class VisualWindow(QMainWindow):
    """Вікно для відображення графіків та хмари слів"""
    def __init__(self, title, parent=None):
        super().__init__(parent)
        self.setWindowTitle(title)
        self.resize(900, 600)
        self.central_widget = QWidget()
        self.setCentralWidget(self.central_widget)
        self.layout = QVBoxLayout(self.central_widget)

    def set_canvas(self, fig):
        self.canvas = FigureCanvas(fig)
        self.layout.addWidget(self.canvas)
        self.canvas.draw()

class CorpusProApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Corpus Pro v14 | Повний професійний функціонал")
        self.resize(1450, 900)
        self.base_path = "CORPUS_LOCAL_DATA"
        self.current_data = []
        self.match_indices = []
        self.total_tokens_count = 0
        self.init_ui()
        self.scan_folders()

    def init_ui(self):
        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)

        # --- ВЕРХНЯ ПАНЕЛЬ КЕРУВАННЯ ---
        top_panel = QFrame()
        top_panel.setStyleSheet("background: #f8f9fa; border-bottom: 2px solid #dee2e6;")
        top_layout = QHBoxLayout(top_panel)

        self.cat_combo = QComboBox()
        self.cat_combo.setMinimumWidth(200)
        self.cat_combo.currentTextChanged.connect(self.load_category_data)

        self.pos_filter = QComboBox()
        self.pos_filter.addItems(["Всі POS", "NOUN", "VERB", "ADJ", "ADV", "PRON", "PROPN", "ADP", "DET", "NUM"])

        self.search_input = QLineEdit()
        self.search_input.setPlaceholderText("Введіть слово (наприклад: Ukraine)...")
        self.search_input.returnPressed.connect(self.run_analysis)

        btn_analyze = QPushButton("🚀 ПОВНИЙ АНАЛІЗ")
        btn_analyze.clicked.connect(self.run_analysis)
        btn_analyze.setStyleSheet("background: #0056b3; color: white; font-weight: bold; padding: 10px 20px;")

        top_layout.addWidget(QLabel("<b>📂 Підкорпус:</b>"))
        top_layout.addWidget(self.cat_combo)
        top_layout.addWidget(QLabel("<b>🏷️ POS:</b>"))
        top_layout.addWidget(self.pos_filter)
        top_layout.addWidget(self.search_input)
        top_layout.addWidget(btn_analyze)
        layout.addWidget(top_panel)

        # --- ТАБИ ---
        self.tabs = QTabWidget()

        # ТАБ 1: Конкорданс та Метадані (Splitter)
        self.tab_concordance = QWidget()
        conc_layout = QHBoxLayout(self.tab_concordance)
        self.splitter = QSplitter(Qt.Orientation.Horizontal)

        # Таблиця метаданих (ліворуч)
        self.meta_table = QTableWidget(0, 2)
        self.meta_table.setHorizontalHeaderLabels(["Dublin Core / Meta", "Значення"])
        self.meta_table.horizontalHeader().setSectionResizeMode(QHeaderView.ResizeMode.Stretch)
        self.meta_table.setStyleSheet("background: #fcfcfc;")

        # Таблиця конкордансу (праворуч)
        self.main_table = QTableWidget(0, 5)
        self.main_table.setHorizontalHeaderLabels(["Лівий контекст", "Ключове слово", "POS", "Правий контекст", "Джерело"])
        self.main_table.itemSelectionChanged.connect(self.update_meta_view)
        self.main_table.horizontalHeader().setSectionResizeMode(QHeaderView.ResizeMode.Stretch)

        self.splitter.addWidget(self.meta_table)
        self.splitter.addWidget(self.main_table)
        self.splitter.setSizes([350, 1100])
        conc_layout.addWidget(self.splitter)

        # ТАБ 2: Контекстні N-грами (узгоджені з пошуковим словом)
        self.tab_ngrams = QWidget()
        ngram_layout = QHBoxLayout(self.tab_ngrams)
        self.bigram_table = QTableWidget(0, 2)
        self.trigram_table = QTableWidget(0, 2)
        self.bigram_table.setHorizontalHeaderLabels(["Біграми (Context-Matched)", "Частота"])
        self.trigram_table.setHorizontalHeaderLabels(["Триграми (Context-Matched)", "Частота"])
        for t in [self.bigram_table, self.trigram_table]:
            t.horizontalHeader().setSectionResizeMode(QHeaderView.ResizeMode.Stretch)
            ngram_layout.addWidget(t)

        self.tabs.addTab(self.tab_concordance, "🔍 Конкорданс та Dublin Core")
        self.tabs.addTab(self.tab_ngrams, "📊 Узгоджені N-грами")
        layout.addWidget(self.tabs)

        # --- ПАНЕЛЬ ВІЗУАЛІЗАЦІЇ ТА ЕКСПОРТУ ---
        export_layout = QHBoxLayout()
        self.btn_disp = QPushButton("📈 Графік дисперсії")
        self.btn_cloud = QPushButton("☁️ Хмара (без стоп-слів)")
        self.btn_csv = QPushButton("📥 Експорт результатів (CSV)")

        for b in [self.btn_disp, self.btn_cloud, self.btn_csv]:
            b.setMinimumHeight(40)
            b.clicked.connect(self.handle_visuals if b != self.btn_csv else self.export_csv)
            export_layout.addWidget(b)
        layout.addLayout(export_layout)

        self.status_bar = QLabel("Готово. Оберіть категорію для початку.")
        layout.addWidget(self.status_bar)

    def scan_folders(self):
        """Пошук папок у базовому шляху"""
        if os.path.exists(self.base_path):
            folders = [d for d in os.listdir(self.base_path) if os.path.isdir(os.path.join(self.base_path, d))]
            self.cat_combo.clear()
            self.cat_combo.addItems(sorted(folders))

    def load_category_data(self):
        """Завантаження всіх даних обраної категорії"""
        cat = self.cat_combo.currentText()
        if not cat: return
        self.current_data = []
        p_tag = os.path.join(self.base_path, cat, "TAGGED")
        p_meta = os.path.join(self.base_path, cat, "METADATA")

        if os.path.exists(p_tag):
            files = [f for f in os.listdir(p_tag) if f.endswith("_tagged.txt")]
            for fn in files:
                bid = fn.replace("_tagged.txt", "")
                # Читання тексту
                with open(os.path.join(p_tag, fn), 'r', encoding='utf-8') as f:
                    content = f.read()
                # Читання метаданих (JSON + DC)
                meta = {}
                mj = os.path.join(p_meta, f"{bid}.json")
                if os.path.exists(mj):
                    with open(mj, 'r', encoding='utf-8') as jf:
                        meta = json.load(jf)
                # Читання Dublin Core файлу для повноти
                mdc = os.path.join(p_meta, f"{bid}_dc.txt")
                if os.path.exists(mdc):
                    with open(mdc, 'r', encoding='utf-8') as df:
                        meta["DublinCore_Raw"] = df.read()

                self.current_data.append({"id": bid, "content": content, "meta": meta})

        self.status_bar.setText(f"Категорія '{cat}': завантажено {len(self.current_data)} текстів.")

    def run_analysis(self):
        """Основний двигун аналізу"""
        query = self.search_input.text().lower().strip()
        target_pos = self.pos_filter.currentText()

        self.main_table.setRowCount(0)
        self.match_indices = []
        self.total_tokens_count = 0

        context_bigrams = []
        context_trigrams = []
        cloud_tokens = []

        for doc in self.current_data:
            # Парсинг токенів (Word/POS)
            tokens = [t.rsplit('/', 1) if '/' in t else [t, 'UNK'] for t in doc['content'].split()]
            words = [t[0] for t in tokens]
            tags = [t[1] for t in tokens]

            for i, w in enumerate(words):
                w_low = w.lower()
                tag = tags[i]

                # Логіка фільтрації
                match_word = (not query) or (query in w_low)
                match_pos = (target_pos == "Всі POS") or (tag == target_pos)

                if match_word and match_pos:
                    # 1. Запис у конкорданс
                    r = self.main_table.rowCount()
                    self.main_table.insertRow(r)
                    self.main_table.setItem(r, 0, QTableWidgetItem(" ".join(words[max(0, i-4):i])))
                    self.main_table.setItem(r, 1, QTableWidgetItem(w))
                    self.main_table.setItem(r, 2, QTableWidgetItem(tag))
                    self.main_table.setItem(r, 3, QTableWidgetItem(" ".join(words[i+1:i+5])))
                    self.main_table.setItem(r, 4, QTableWidgetItem(doc['id']))

                    # Глобальний індекс для дисперсії
                    self.match_indices.append(self.total_tokens_count + i)

                    # 2. Формування N-грам (УЗГОДЖЕНИХ з пошуковим словом та БЕЗ стоп-слів)
                    # Біграми (попереднє+поточне або поточне+наступне)
                    if i > 0:
                        prev_w = words[i-1]
                        if prev_w.lower() not in STOP_WORDS and w_low not in STOP_WORDS:
                            context_bigrams.append(f"{prev_w} {w}")
                    if i < len(words) - 1:
                        next_w = words[i+1]
                        if w_low not in STOP_WORDS and next_w.lower() not in STOP_WORDS:
                            context_bigrams.append(f"{w} {next_w}")

                    # Триграми (центровані на пошуковому слові)
                    if i > 0 and i < len(words) - 1:
                        prev_w, next_w = words[i-1], words[i+1]
                        if prev_w.lower() not in STOP_WORDS and w_low not in STOP_WORDS and next_w.lower() not in STOP_WORDS:
                            context_trigrams.append(f"{prev_w} {w} {next_w}")

                # Збір для хмари слів (тільки значущі слова)
                if w_low not in STOP_WORDS and len(w_low) > 2:
                    cloud_tokens.append(w_low)

            self.total_tokens_count += len(words)

        # Оновлення інтерфейсу
        self.update_stats_table(self.bigram_table, context_bigrams)
        self.update_stats_table(self.trigram_table, context_trigrams)
        self.all_cloud_text = " ".join(cloud_tokens)
        self.status_bar.setText(f"Аналіз завершено. Знайдено збігів: {len(self.match_indices)} у {self.total_tokens_count} токенах.")

    def update_stats_table(self, table, data):
        """Заповнення таблиць N-грам"""
        table.setRowCount(0)
        counts = Counter(data).most_common(50)
        for i, (txt, count) in enumerate(counts):
            table.insertRow(i)
            table.setItem(i, 0, QTableWidgetItem(txt))
            table.setItem(i, 1, QTableWidgetItem(str(count)))

    def update_meta_view(self):
        """Відображення метаданих для обраного рядка конкордансу"""
        items = self.main_table.selectedItems()
        if not items: return
        row = items[0].row()
        doc_id = self.main_table.item(row, 4).text()

        doc = next((d for d in self.current_data if d['id'] == doc_id), None)
        self.meta_table.setRowCount(0)

        if doc and doc['meta']:
            # Спочатку виводимо ключі з JSON
            row_idx = 0
            for k, v in doc['meta'].items():
                if k == "DublinCore_Raw": continue
                self.meta_table.insertRow(row_idx)
                self.meta_table.setItem(row_idx, 0, QTableWidgetItem(str(k).upper()))
                self.meta_table.setItem(row_idx, 1, QTableWidgetItem(str(v)))
                row_idx += 1

            # Додаємо сирий DC текст як примітку
            if "DublinCore_Raw" in doc['meta']:
                self.meta_table.insertRow(row_idx)
                self.meta_table.setItem(row_idx, 0, QTableWidgetItem("DC_FULL_TEXT"))
                self.meta_table.setItem(row_idx, 1, QTableWidgetItem("Див. вміст нижче..."))
                QToolTip.showText(Qt.Cursor.pos(), doc['meta']["DublinCore_Raw"])

    def handle_visuals(self):
        """Обробка кнопок візуалізації"""
        btn_text = self.sender().text()

        if "Дисперсія" in btn_text:
            if not self.match_indices:
                return QMessageBox.information(self, "Інфо", "Немає даних для побудови графіка.")
            fig, ax = plt.subplots(figsize=(12, 3))
            ax.vlines(self.match_indices, 0, 1, colors='#d93025', linewidth=0.5)
            ax.set_xlim(0, self.total_tokens_count)
            ax.set_title(f"Розподіл запиту '{self.search_input.text()}' у корпусі")
            ax.set_xlabel("Позиція токена")
            ax.set_yticks([])
            self.v_win = VisualWindow("Дисперсійний аналіз")
            self.v_win.set_canvas(fig)
            self.v_win.show()

        elif "Хмара" in btn_text:
            if not hasattr(self, 'all_cloud_text') or not self.all_cloud_text:
                return QMessageBox.warning(self, "Увага", "Спочатку запустіть аналіз.")
            wc = WordCloud(width=1000, height=600, background_color='white', colormap='tab10').generate(self.all_cloud_text)
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.imshow(wc, interpolation='bilinear')
            ax.axis('off')
            ax.set_title("Хмара значущих слів (без стоп-слів)")
            self.v_win = VisualWindow("Семантична хмара")
            self.v_win.set_canvas(fig)
            self.v_win.show()

    def export_csv(self):
        """Експорт поточної таблиці конкордансу"""
        if self.main_table.rowCount() == 0:
            return QMessageBox.warning(self, "Помилка", "Немає даних для експорту.")

        path, _ = QFileDialog.getSaveFileName(self, "Зберегти CSV", "", "CSV Files (*.csv)")
        if path:
            try:
                with open(path, 'w', newline='', encoding='utf-8-sig') as f:
                    writer = csv.writer(f)
                    writer.writerow(["L-Context", "Keyword", "POS", "R-Context", "Source_ID"])
                    for r in range(self.main_table.rowCount()):
                        row_data = [self.main_table.item(r, c).text() for c in range(5)]
                        writer.writerow(row_data)
                QMessageBox.information(self, "Успіх", "Файл успішно збережено.")
            except Exception as e:
                QMessageBox.critical(self, "Помилка", f"Не вдалося зберегти файл: {e}")

if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    # Встановлення шрифту для підтримки кирилиці та читабельності
    app.setFont(QFont("Segoe UI", 10))
    window = CorpusProApp()
    window.show()
    sys.exit(app.exec())